# Stereo and Depth Vision

> **Advanced · 3D geometry**


## Why this matters

Stereo estimates depth from parallax; depth processing turns those estimates into useful 3D points. Both require calibrated geometry and careful treatment of uncertainty.

**Where it appears:** Robot perception, depth cameras, scene measurement, obstacle cues, and point-cloud visualization.


## Learning Objectives

- Understand the stereo correspondence problem and epipolar geometry basics
- Compute a disparity map from a rectified stereo pair
- Convert disparity to approximate depth given calibration parameters
- Convert a depth map into a 3D point cloud using camera intrinsics
- Filter and clean noisy depth data before using it
- Visualize point clouds and understand basic 3D data structures


## Prerequisites

19 Camera Calibration and 6DoF Object Pose

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

rectification, `StereoBM`/`StereoSGBM`, disparity, back-projection, point clouds, depth filtering

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Stereo Vision

Stereo vision recovers depth by finding, for each pixel in a left image,
its corresponding pixel in a right image taken from a known, horizontally
-offset viewpoint. For a **rectified** stereo pair (rows aligned so
correspondences lie on the same horizontal line), depth is inversely
proportional to **disparity** (the horizontal pixel offset between
corresponding points): `depth = (focal_length * baseline) / disparity`.
This notebook builds a synthetic rectified stereo pair with known ground-
truth disparity to validate the pipeline end-to-end.


### 3D Vision and Depth Processing

Given a depth map (from stereo, notebook 30, or a depth sensor) and known
camera intrinsics, each pixel `(u, v)` with depth `Z` back-projects to a 3D
point via the pinhole model: `X = (u - cx) * Z / fx`, `Y = (v - cy) * Z /
fy`. Real depth data is noisy at object edges and in low-texture regions,
so cleaning (removing invalid/zero depth, outlier filtering) is a required
step before any downstream 3D processing (mesh reconstruction, obstacle
detection, AR placement).


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Stereo Vision


### 1. Building a synthetic rectified stereo pair

Render a scene with objects at different depths, then create the 'right' view by shifting each object horizontally by an amount inversely proportional to its assigned depth -- exactly what a real stereo camera pair produces.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid

left = load_real_image("images/stereo", "aloeL.jpg")
right = load_real_image("images/stereo", "aloeR.jpg")

# Resize for faster processing in notebook
h, w = left.shape[:2]
scale = 0.5
left = cv2.resize(left, (int(w * scale), int(h * scale)))
right = cv2.resize(right, (int(w * scale), int(h * scale)))

show_grid([("Left camera view", left), ("Right camera view", right)])

### 2. Computing the disparity map

`cv2.StereoSGBM` (semi-global block matching) computes per-pixel disparity from grayscale rectified images -- parameters must be tuned to the scene's disparity range.


In [ ]:
def compute_disparity_map(
    left: np.ndarray, right: np.ndarray, num_disparities=64, block_size=15
) -> np.ndarray:
    left_gray = cv2.cvtColor(left, cv2.COLOR_BGR2GRAY)
    right_gray = cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)

    stereo = cv2.StereoBM_create(numDisparities=num_disparities, blockSize=block_size)
    raw_disparity = stereo.compute(left_gray, right_gray).astype(np.float32) / 16.0
    return raw_disparity


disparity = compute_disparity_map(left, right, num_disparities=64, block_size=15)

disparity_vis = cv2.normalize(
    np.clip(disparity, 0, None), None, 0, 255, cv2.NORM_MINMAX
).astype(np.uint8)

show_grid([("Computed Disparity (brighter = closer)", disparity_vis)])

### 3. Converting disparity to depth

Given known camera focal length and stereo baseline (from calibration in the previous notebook), convert disparity at each ground-truth object center to an actual depth estimate, and sanity-check against the known relative ordering.


In [ ]:
def disparity_to_depth(
    disparity_px: float, focal_length_px: float = 700, baseline_mm: float = 60
) -> float:
    if disparity_px <= 0:
        return float("inf")
    return (focal_length_px * baseline_mm) / disparity_px


print("Estimated depth per object (mm) from Aloe dataset:")
# Sample points: (cx, cy) and a description
sample_points = [
    ((250, 250), "Foreground Plant"),
    ((100, 100), "Background Area"),
    ((150, 400), "Left side pot"),
]

for (cx, cy), desc in sample_points:
    measured_disp = disparity[cy, cx]
    depth_mm = disparity_to_depth(measured_disp)
    print(
        f"  {desc} at {(cx, cy)}: "
        f"measured_disparity={measured_disp:.1f}px, estimated_depth={depth_mm:.0f}mm"
    )

## Part 2: 3D Vision and Depth Processing


### 1. Generating a synthetic depth map

Build a depth map with a clear geometric structure (a tilted plane + a raised block) plus injected sensor-like noise and dropout, to have realistic data to clean.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid


def compute_real_depth_map() -> np.ndarray:
    """Compute a real depth map from a stereo pair."""
    left = load_real_image("images/stereo", "aloeL.jpg", cv2.IMREAD_GRAYSCALE)
    right = load_real_image("images/stereo", "aloeR.jpg", cv2.IMREAD_GRAYSCALE)
    # Resize for faster processing
    left = cv2.resize(left, (0, 0), fx=0.5, fy=0.5)
    right = cv2.resize(right, (0, 0), fx=0.5, fy=0.5)

    stereo = cv2.StereoBM_create(numDisparities=64, blockSize=15)
    disparity = stereo.compute(left, right).astype(np.float32) / 16.0

    # Convert disparity to depth (Depth = Constant / Disparity)
    depth = np.zeros_like(disparity)
    valid = disparity > 0
    depth[valid] = 20000.0 / disparity[valid]  # scale roughly to millimeters
    depth[~valid] = 0  # 0 indicates invalid depth

    return depth


depth_map = compute_real_depth_map()

# Visualize depth map (closer = brighter)
depth_vis = cv2.normalize(
    np.clip(1.0 / (depth_map + 1e-5), 0, None), None, 0, 255, cv2.NORM_MINMAX
).astype(np.uint8)

show_grid([("Raw Depth Map (real stereo data)", depth_vis)])

### 2. Cleaning invalid/noisy depth

Remove zero/invalid pixels and smooth remaining noise with a median filter, which (as covered in notebook 10) is well-suited to spike-like sensor noise while preserving the underlying structure.


In [ ]:
def clean_depth(depth: np.ndarray, max_valid: float = 3000) -> np.ndarray:
    cleaned = depth.copy()
    invalid = (cleaned <= 0) | (cleaned > max_valid)
    cleaned[invalid] = np.nan
    filled = cv2.inpaint(
        np.nan_to_num(cleaned).astype(np.float32),
        invalid.astype(np.uint8),
        inpaintRadius=3,
        flags=cv2.INPAINT_TELEA,
    )
    smoothed = cv2.medianBlur(filled.astype(np.float32), 5)
    return smoothed


cleaned_depth = clean_depth(depth_map)
invalid_before = int((depth_map <= 0).sum())
print(f"Invalid/dropout pixels before cleaning: {invalid_before}")
print(f"Invalid pixels after cleaning: {int((cleaned_depth <= 0).sum())}")

### 3. Back-projecting to a 3D point cloud

Use the pinhole back-projection formula to convert the cleaned depth map into an (N, 3) array of 3D points, then visualize a top-down (bird's-eye) view -- a simple but genuinely informative 3D sanity check that doesn't require a 3D viewer.


In [ ]:
def depth_to_pointcloud(
    depth: np.ndarray, fx=500, fy=500, cx=None, cy=None
) -> np.ndarray:
    h, w = depth.shape
    cx = cx if cx is not None else w / 2
    cy = cy if cy is not None else h / 2
    us, vs = np.meshgrid(np.arange(w), np.arange(h))
    Z = depth
    X = (us - cx) * Z / fx
    Y = (vs - cy) * Z / fy
    points = np.stack([X, Y, Z], axis=-1).reshape(-1, 3)
    return points[points[:, 2] > 0]  # drop remaining invalid points


points = depth_to_pointcloud(cleaned_depth)
print(f"Point cloud size: {points.shape[0]} points")

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.scatter(
    points[::20, 0], points[::20, 2], s=1
)  # top-down view: X (left-right) vs Z (depth)
plt.xlabel("X (mm)")
plt.ylabel("Z / depth (mm)")
plt.title("Top-down view of point cloud (block should appear as a closer cluster)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Stereo Vision: Generating Red-Cyan Anaglyph Images

Red-cyan anaglyphs display stereo depth by encoding the Left view in the Red channel and the Right view in the Blue and Green (Cyan) channels. When viewed with 3D glasses, this creates depth perception.


In [ ]:
# Create mock left/right rectified views
img = load_real_image("images/landscapes", "city.jpg")
h, w = img.shape[:2]

# Left view: original image
left = img.copy()
# Right view: shift pixels horizontally to simulate disparity
right = np.zeros_like(img)
right[:, 5:] = img[:, :-5]

# Extract channels
red_channel = left[:, :, 2]  # Red from Left view
green_channel = right[:, :, 1]  # Green from Right view
blue_channel = right[:, :, 0]  # Blue from Right view

# Merge to form red-cyan anaglyph
anaglyph = cv2.merge([blue_channel, green_channel, red_channel])

print("Red-Cyan Anaglyph image generated successfully.")
show(anaglyph, "Synthesized 3D Anaglyph View")

### Mini Project — 3D Vision and Depth Processing: Edge-Preserving Bilateral Depth Filtering

Depth maps produced by stereo matching or sensor devices contain high-frequency noise and missing values along edges. To clean this, we apply a bilateral filter that smooths noise while keeping sharp boundary margins.


In [ ]:
# Generate a noisy synthetic depth map
depth = depth_map.copy()
# Add standard Gaussian noise
rng = np.random.default_rng(42)
noisy_depth = depth + rng.normal(0, 15, depth.shape)
noisy_depth = np.clip(noisy_depth, 0, 255).astype(np.uint8)

# Apply Bilateral filter to smooth noise while keeping boundaries sharp
filtered_depth = cv2.bilateralFilter(noisy_depth, 9, 50, 50)

print("Bilateral depth filtering complete.")
show_grid(
    [
        ("Noisy Sensor Depth", noisy_depth),
        ("Edge-Preserved Filtered Depth", filtered_depth),
    ]
)

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Stereo Vision
1. Sweep `numDisparities` and `blockSize` and observe the effect on disparity map smoothness vs detail.
2. Add a fourth object with an even smaller disparity (farther away) and confirm depth ordering is still correct.
3. Research (markdown) what stereo RECTIFICATION (`cv2.stereoRectify`) does and why it's a prerequisite for this whole pipeline on real camera pairs.

Use the empty cell below to work through them.


#### Solutions — Stereo Vision

In [ ]:
# Solution 1: Sweep numDisparities and blockSize
# Explanation: `numDisparities` defines the depth search range (must be divisible by 16).
# Sweeping it determines if the system can resolve nearby depth layers. `blockSize` sets
# local matching resolution. Small `blockSize` (e.g. 5) resolves fine edges but creates noise
# in flat textureless areas. Large `blockSize` (e.g. 21) reduces noise but blurs depth margins.


In [ ]:
# Solution 2: Verify depth ordering with a fourth distant object
# Disparity is inversely proportional to depth: d = (f * B) / Z. A fourth object placed
# farther away than the existing three will produce a smaller disparity. Confirming this
# smaller disparity maps to a larger distance validates the depth ordering calculations.


In [ ]:
# Solution 3: What stereo rectification does and why it is a prerequisite
# Stereo rectification warp-aligns Left and Right camera planes so that epipolar lines run
# perfectly parallel and horizontal. In rectified images, matching features share the same
# vertical Y-coordinate (pixel search is simplified to a 1D horizontal line scan instead of
# a 2D area scan), reducing search times and computational complexity.


### Exercises — 3D Vision and Depth Processing
1. Add a second raised block at a different depth and confirm it's visible as a separate cluster in the top-down plot.
2. Implement simple voxel downsampling: round point coordinates to a grid and deduplicate to reduce point count.
3. Export the point cloud to a `.ply` file (a simple text format) that could be opened in MeshLab/CloudCompare.

Use the empty cell below to work through them.


#### Solutions — 3D Vision and Depth Processing

In [ ]:
# Solution 1: Add a second raised block at different depth
# In a point cloud cluster plot, adding a second raised block at a different Z-coordinate
# will produce a distinct point group. Projecting these points onto a top-down scatter plot (X-Y plane)
# and coloring by Z-height reveals two separate clusters, validating the depth segmentation.


In [ ]:
# Solution 2: Voxel grid downsampling
def voxel_downsample(points: np.ndarray, voxel_size: float = 0.05) -> np.ndarray:
    """Reduce point cloud density by grouping points into voxel grid bins."""
    # Group coordinates into grid indices
    grid_coords = np.floor(points / voxel_size).astype(np.int32)
    # Get unique voxel indices
    _, idxs = np.unique(grid_coords, axis=0, return_index=True)
    return points[idxs]


# Verify downsampler
pts = np.random.rand(100, 3)
downsampled = voxel_downsample(pts, 0.2)
print(f"Original points count: 100 | Voxel downsampled count: {len(downsampled)}")

In [ ]:
# Solution 3: Export point cloud to PLY format
def export_to_ply(filepath: str, points: np.ndarray) -> None:
    """Write a 3D point cloud array to a PLY text file format."""
    with open(filepath, "w") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {len(points)}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        f.write("end_header\n")
        for p in points:
            f.write(f"{p[0]:.4f} {p[1]:.4f} {p[2]:.4f}\n")

## Summary

You can inspect a disparity map, convert valid disparity to depth, and back-project a depth image while recognizing where reconstruction is unreliable.

- **Best Practices:** Rectify before matching, mask invalid disparity, keep units explicit, and visualize depth distributions before making geometric claims.
- **Common Pitfalls:** Using unrectified pairs, interpreting zero/negative disparity as a valid object, and forgetting that depth uncertainty grows with distance.